In [1]:
import os

from dotenv import find_dotenv, load_dotenv
from google import genai
from qdrant_client import QdrantClient

load_dotenv(find_dotenv(usecwd=True))
api_key = os.getenv("GOOGLE_API_KEY")
if not api_key:
    raise ValueError("Set GOOGLE_API_KEY in a .env file or the environment.")

gemini_client = genai.Client(api_key=api_key)

In [2]:
from google import genai
from qdrant_client import QdrantClient

In [3]:
def get_embedding(text: str, model: str = "gemini-embedding-001") -> list[float]:
    response = gemini_client.models.embed_content(
        model=model,
        contents=text,
    )
    return response.embeddings[0].values

In [4]:
qdrant_client = QdrantClient(url="http://localhost:6333")

In [5]:
def retrieve_data(query, qdrant_client, k=5):

    query_embedding = get_embedding(query)

    results = qdrant_client.query_points(
        collection_name="Amazon-items-collection-00",
        query=query_embedding,
        limit=k,
    )

    retrieved_context_ids = []
    retrieved_context = []
    similarity_scores = []
    retrieved_context_ratings = []

    for result in results.points:
        retrieved_context_ids.append(result.payload["parent_asin"])
        retrieved_context.append(result.payload["description"])
        retrieved_context_ratings.append(result.payload["average_rating"])
        similarity_scores.append(result.score)

    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_context": retrieved_context,
        "retrieved_context_ratings": retrieved_context_ratings,
        "similarity_scores": similarity_scores,
    }

In [6]:
retrieved_context = retrieve_data("What kind of earphones can I get ?", qdrant_client, k=10)

In [7]:
retrieved_context

{'retrieved_context_ids': ['B0C52WNWY6',
  'B0B3J1NNGL',
  'B09Y57K53R',
  'B0B51PZC3L',
  'B0BXRMTZCD',
  'B0C3GP9QFB',
  'B0CGHG8Y7H',
  'B0C999S8YM',
  'B0BM961XHH',
  'B09NKXF9YQ'],
 'retrieved_context': ["Wireless Earbud, Bluetooth Headphones in Ear Noise Cancelling Mic Mini Ear Buds, Bluetooth 5.1 Light Weight Deep Bass, IP7 Waterproof Wireless Headphones, 30H Playtime for Workout/Home/Office, White CVC8.0 Noise Reduction and Hi-Fi Stereo Sound: Adopt advanced Bluetooth chip and high-sensitivity diaphragm and filter out outside noise allowing the other side to hear you more clearly. Besides, the built-in high-quality microphone can provide you with authentic sound and powerful bass. Hear your original voice whether you video with friends or make phone calls. Great Battery Life and Dual LED Power Display: The wireless earbuds can last 6-7 hours of high quality sound output on a single charge. Features with smart LED digital display can help check the power of charging case at any 

In [8]:
def process_context(context):
    formatted_context = ""

    for id, chunk, rating in zip(
        context["retrieved_context_ids"],
        context["retrieved_context"],
        context["retrieved_context_ratings"],
    ):
        formatted_context += (
            f"- ID: {id}, rating: {rating}, description: {chunk}\n"
        )

    return formatted_context

In [9]:
preprocessed_context = process_context(retrieved_context)

In [10]:
print(preprocessed_context)

- ID: B0C52WNWY6, rating: 4.3, description: Wireless Earbud, Bluetooth Headphones in Ear Noise Cancelling Mic Mini Ear Buds, Bluetooth 5.1 Light Weight Deep Bass, IP7 Waterproof Wireless Headphones, 30H Playtime for Workout/Home/Office, White CVC8.0 Noise Reduction and Hi-Fi Stereo Sound: Adopt advanced Bluetooth chip and high-sensitivity diaphragm and filter out outside noise allowing the other side to hear you more clearly. Besides, the built-in high-quality microphone can provide you with authentic sound and powerful bass. Hear your original voice whether you video with friends or make phone calls. Great Battery Life and Dual LED Power Display: The wireless earbuds can last 6-7 hours of high quality sound output on a single charge. Features with smart LED digital display can help check the power of charging case at any time, so you always know when it's time to recharge.Take it with you, you can relax enjoy music while traveling. IP7 Waterproof Earbuds and Ergonomic Design: Bluetoot

In [11]:
def build_prompt(preprocessed_context, question):
    
    prompt = f"""
You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructtions:
- You need to answer the question based on the provided context only.
- Never use word context and refer to it as the available products.

Context:
{preprocessed_context}

Question:
{question}
"""
    
    return prompt

In [12]:
prompt = build_prompt(preprocessed_context, "What kind of earphones can I get?")

In [13]:
print(prompt )


You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructtions:
- You need to answer the question based on the provided context only.
- Never use word context and refer to it as the available products.

Context:
- ID: B0C52WNWY6, rating: 4.3, description: Wireless Earbud, Bluetooth Headphones in Ear Noise Cancelling Mic Mini Ear Buds, Bluetooth 5.1 Light Weight Deep Bass, IP7 Waterproof Wireless Headphones, 30H Playtime for Workout/Home/Office, White CVC8.0 Noise Reduction and Hi-Fi Stereo Sound: Adopt advanced Bluetooth chip and high-sensitivity diaphragm and filter out outside noise allowing the other side to hear you more clearly. Besides, the built-in high-quality microphone can provide you with authentic sound and powerful bass. Hear your original voice whether you video with friends or make phone calls. Great Battery Life and Dual LED Power Display: The wireless earbuds can last 6-7 hours of

In [14]:
from google.genai import types

def generate_answer(prompt: str, model: str = "gemini-3.8-flash") -> str:
    response = gemini_client.models.generate_content(
        model=model,
        contents=prompt,
        config=types.GenerateContentConfig(
            thinking_config=types.ThinkingConfig(thinking_level="low"),
        ),
    )
    return response.text

In [15]:
print(generate_answer(prompt))

Based on the available products, here are the kinds of earphones and headphones you can get:

1. **In-Ear Wireless Earbuds with Charging Case (ID: B0C52WNWY6):**
   * Features Bluetooth 5.1, CVC8.0 noise reduction, Hi-Fi stereo sound, deep bass, IP7 waterproofing, dual LED power display, touch controls, and up to 30 hours of playtime.

2. **True Wireless Sleep Earbuds (ID: B0B3J1NNGL):**
   * Tiny, lightweight (0.095 oz per earbud) in-ear headphones designed for sleeping, fitting comfortably even for side sleepers. Blocks outside noise by up to 24dB and provides 4.5 hours of playback (27 additional hours with the charging case).

3. **Bone Conduction Ear Hook Headsets (ID: B09Y57K53R):**
   * Non-in-ear/non-ear-plug hanging design using bone conduction to keep your ears open and avoid swelling or pain. Features Bluetooth 5.0, a built-in microphone, waterproof/sweatproof design, and up to 10 hours of playback.

4. **Sport Wireless Earbuds with Ear Hooks (ID: B0B51PZC3L):**
   * Designed

In [16]:
def rag_pipeline(question, top_k=5):
    
    qdrant_client = QdrantClient(url="http://localhost:6333")
    
    retrieved_context = retrieve_data(question, qdrant_client, top_k)
    preprocessed_context = process_context(retrieved_context)
    prompt = build_prompt(preprocessed_context, question)
    answer = generate_answer(prompt)
    
    return answer

In [17]:
print(rag_pipeline("What kind of earphones can I get with ratings above 4.5?"))

Based on the available products, there are no earphones or headphones with a rating strictly above 4.5. 

The highest-rated option in stock is rated at 4.5:
* **ASMRBAND Headband Headphones (Rating: 4.5, ID: B0BXRMTZCD)**: Adjustable sleep headphones integrated into a comfortable, soft fabric headband featuring ultra-thin speakers, Wireless 5.2 connectivity, a built-in microphone, and 10+ hours of battery life.
